# Instrumented Principal Components - US Equities Panel

This notebook generates walk-forward validation predictions from instrumented principal
components. Readers choose the labels, factor count, parameter overrides, and execution tier.
Shared latent-factor code owns fold preparation, train-only fitting, convergence validation,
fitted-state persistence, prediction reconstruction, exact coverage, metrics, and registry
writes.

The implementation is in `case_studies/utils/latent_factors/ipca.py` and its shared research
adapter. Readers can modify the factor construction in ordinary Python while preserving the same
request, result, and catalog boundary.

**Learning objectives**

- Configure label-specific IPCA requests with train-only fold fitting.
- Trace characteristic loadings, convergence evidence, and reconstructed predictions.
- Validate exact coverage before publishing compatible label sets.

**Book reference**: Chapter 13

**Prerequisites**: `05_evaluation.py` and the finalized label and feature artifacts.

In [1]:
"""Generate IPCA validation predictions through the shared research interface."""

import os
from pathlib import Path

import polars as pl
import yaml

from case_studies.research import Study, open_study, plan_models
from utils.modeling import load_configs
from utils.paths import REPO_ROOT, get_case_study_dir

In [2]:
CASE_STUDY_ID = "us_equities_panel"
LABELS = []
N_FACTORS = 5
OVERRIDES = {}
EXECUTION_TIER = "canonical"
WORKSPACE = "experiments"
MAX_SYMBOLS = 0
FOLD_IDS = []
PREVIEW_N_FACTORS = 0
PREVIEW_MAX_ITER = 0

## Configure the experiment

`LABELS = []` runs the primary label and every configured variant. Set it to a visible subset for
a targeted experiment. `N_FACTORS` and `OVERRIDES` are resolved into the complete model
specification. Canonical execution uses the complete panel, fold protocol, and convergence
settings. Reduced checks use the preview tier, declare every reduction below, and remain outside
official result populations.

In [4]:
case_dir = get_case_study_dir(CASE_STUDY_ID)
setup = yaml.safe_load((case_dir / "config" / "setup.yaml").read_text())
published_labels = [setup["labels"]["primary"], *setup["labels"].get("variants", [])]
selected_labels = list(LABELS) if LABELS else published_labels
unknown_labels = sorted(set(selected_labels) - set(published_labels))
if unknown_labels:
    raise ValueError(f"Unknown labels: {unknown_labels}")
if len(selected_labels) != len(set(selected_labels)):
    raise ValueError("LABELS contains duplicates")

for label in selected_labels:
    configured = {
        config["config_name"]
        for config in load_configs(CASE_STUDY_ID, label, family="latent_factors")
    }
    if "ipca" not in configured:
        raise ValueError(f"IPCA is not configured for {label}")

label_menu = pl.DataFrame(
    {
        "label": published_labels,
        "selected": [label in selected_labels for label in published_labels],
        "n_factors": [N_FACTORS] * len(published_labels),
    }
)
label_menu

Case study: us_equities_panel
Model: ipca
Primary label: fwd_ret_1d
Variant labels: ['fwd_ret_5d', 'fwd_ret_21d']
Dataset rows: 9,205,450
Features: 72
Splits: 16


In [ ]:
preview_reductions = {}
if MAX_SYMBOLS:
    preview_reductions["max_symbols"] = int(MAX_SYMBOLS)
if FOLD_IDS:
    preview_reductions["folds"] = [int(fold) for fold in FOLD_IDS]
if PREVIEW_N_FACTORS:
    preview_reductions["n_factors"] = int(PREVIEW_N_FACTORS)
if PREVIEW_MAX_ITER:
    preview_reductions["max_iter"] = int(PREVIEW_MAX_ITER)

# Both tiers resolve the study through `open_study`, never `Study.open`/`Study.regenerate`
# directly. In a maintainer worktree the generated directories are symlinks to shared data, and
# `open_study` handles that by reading inputs in place - `root` stays the release case directory
# and only writes are redirected to the workspace. `Study.open(workspace=...)` instead puts `root`
# inside the workspace, so `source = self.root / "labels"` (workspace.py:274) resolves somewhere
# else and `_ensure_input_link` rejects the link a sibling notebook already made. Two notebooks in
# one session then cannot both open a preview workspace.
if EXECUTION_TIER == "canonical":
    if preview_reductions:
        raise ValueError("Canonical execution cannot declare preview reductions")
    study = open_study(CASE_STUDY_ID, execution_tier=EXECUTION_TIER)
elif EXECUTION_TIER == "preview":
    if not preview_reductions:
        raise ValueError("Preview execution requires at least one declared reduction")
    study = open_study(
        CASE_STUDY_ID,
        execution_tier=EXECUTION_TIER,
        workspace=Path(os.environ.get("ML4T_OUTPUT_DIR") or WORKSPACE),
    )
else:
    raise ValueError("EXECUTION_TIER must be 'canonical' or 'preview'")

## Build the label requests

Each selected label receives a separate IPCA request under the shared factor-count specification.

In [5]:
requests = tuple(
    study.model(
        family="latent_factors",
        label=label,
        config_name="ipca",
        overrides={"n_factors": int(N_FACTORS), **OVERRIDES},
        execution_tier=EXECUTION_TIER,
        preview_reductions=preview_reductions,
    )
    for label in selected_labels
)

request_table = pl.DataFrame(
    {
        "family": [request.family for request in requests],
        "label": [request.label for request in requests],
        "config_name": [request.config_name for request in requests],
        "overrides": [str(request.overrides) for request in requests],
        "execution_tier": [request.execution_tier.value for request in requests],
        "preview_reductions": [str(request.preview_reductions) for request in requests],
    }
)
request_table

Latent factor CV: 1 models × 16 folds
Log file: case_studies/us_equities_panel/run_log/latent_factors.log
Scoring: dates=rebalance cadence=daily_close step=1 checkpoint_selection=fixed reporting_epoch=last


  ipca (K=5):


    Fold 0: ragged train=2519, val=252, max_N=2684


      fold 0: reported_epoch=0, IC=-0.0008, 92.3s


    Fold 1: ragged train=2519, val=252, max_N=2684


...

    Fold 14: ragged train=2519, val=252, max_N=1084


      fold 14: reported_epoch=0, IC=+0.0276, 31.2s


    Fold 15: ragged train=2493, val=252, max_N=987


      fold 15: reported_epoch=0, IC=-0.0040, 27.4s


    -> best epoch=0, IC=+0.0049 (1285.5s)
  Best: ipca (IC=+0.0049)
[{'model_name': 'ipca', 'mean_ic': 0.0049, 'best_epoch': 0, 'n_folds': 16, 'elapsed_s': 1285.5, 'started_at': '2026-04-28T02:32:19.742134+00:00'}]
shape: (16, 6)
┌─────────┬───────┬─────────┬─────────┬────────┬────────────────┐
│ fold_id ┆ epoch ┆ ic_mean ┆ n_train ┆ n_test ┆ n_scored_dates │
│ ---     ┆ ---   ┆ ---     ┆ ---     ┆ ---    ┆ ---            │
│ i64     ┆ i64   ┆ f64     ┆ i64     ┆ i64    ┆ i64            │
╞═════════╪═══════╪═════════╪═════════╪════════╪════════════════╡
│ 0       ┆ 0     ┆ -0.0008 ┆ 2519    ┆ 252    ┆ 252            │
│ 1       ┆ 0     ┆ -0.0105 ┆ 2519    ┆ 252    ┆ 248            │
│ 2       ┆ 0     ┆ 0.0028  ┆ 2519    ┆ 252    ┆ 249            │
│ 3       ┆ 0     ┆ 0.0008  ┆ 2519    ┆ 252    ┆ 246            │
│ 4       ┆ 0     ┆ -0.0095 ┆ 2519    ┆ 252    ┆ 252            │
│ …       ┆ …     ┆ …       ┆ …       ┆ …      ┆ …              │
│ 11      ┆ 0     ┆ 0.0145  ┆ 2519    ┆ 252 

## Plan and execute the selected labels

The planner resolves every label-specific training and checkpoint identity before fitting and
writes the canonical checkpoint population first. Each fold then fits IPCA on its training panel
only. The runner validates convergence, persists the
fitted factor state, and reconstructs each registered prediction set from those artifacts before
accepting cached work.

In [6]:
plan = plan_models(study, requests=requests)
official_population = None
if EXECUTION_TIER == "canonical":
    official_population = plan.create_population(
        name="us-equities-ipca-checkpoints-v1",
    )

planned_population = pl.DataFrame(
    {
        "label": [member.label for member in plan.members],
        "config_name": [member.config_name for member in plan.members],
        "checkpoint_kind": [member.checkpoint_kind for member in plan.members],
        "checkpoint_value": [member.checkpoint_value for member in plan.members],
        "training_hash": [member.training_hash for member in plan.members],
        "prediction_hash": [member.prediction_hash for member in plan.members],
    }
)
planned_population

Latent factor CV: 1 models × 16 folds
Log file: case_studies/us_equities_panel/run_log/latent_factors.log
Scoring: dates=rebalance cadence=daily_close step=5 checkpoint_selection=fixed reporting_epoch=last
  ipca (K=5):


    Fold 0: ragged train=2515, val=252, max_N=2678


      fold 0: reported_epoch=0, IC=+0.0075, 92.6s


    Fold 1: ragged train=2515, val=252, max_N=2678


      fold 1: reported_epoch=0, IC=-0.0206, 87.8s


...

    Fold 14: ragged train=2499, val=252, max_N=1083


      fold 14: reported_epoch=0, IC=-0.0071, 29.6s


    Fold 15: ragged train=2472, val=252, max_N=987


      fold 15: reported_epoch=0, IC=+0.0288, 26.4s


    -> best epoch=0, IC=+0.0181 (1251.4s)
  Best: ipca (IC=+0.0181)
fwd_ret_5d [{'model_name': 'ipca', 'mean_ic': 0.0138, 'best_epoch': 0, 'n_folds': 16, 'elapsed_s': 1282.2, 'started_at': '2026-04-28T02:53:52.727272+00:00'}]
fwd_ret_21d [{'model_name': 'ipca', 'mean_ic': 0.0181, 'best_epoch': 0, 'n_folds': 16, 'elapsed_s': 1251.4, 'started_at': '2026-04-28T03:15:22.750315+00:00'}]


In [ ]:
execution = plan.run()

## Inspect the resolved computation

In [ ]:
resolved_rows = []
for run in execution.runs:
    spec = run.training.spec()
    computation = spec["computation"]
    resolved_rows.append(
        {
            "label": spec["label"],
            "features": len(computation["feature_names"]),
            "folds": computation["expected_prediction_keys"]["n_folds"],
            "eligible_rows": computation["expected_prediction_keys"]["n_rows"],
            "n_factors": computation["model"]["n_factors"],
            "max_iter": computation["model"]["params"]["max_iter"],
            "fold_workers": computation["runtime"]["fold_workers"],
            "training_hash": run.training.hash,
        }
    )

resolved_table = pl.DataFrame(resolved_rows).sort("label")
resolved_table

## Validate and inspect the handoff

Each catalog row is a complete validation prediction set with exact training, label, fold, and
fitted-state lineage. Downstream notebooks select these rows with Polars rather than copying
hashes, while the hashes remain visible for exact provenance.

In [ ]:
catalog_columns = [
    "family",
    "config_name",
    "label",
    "split",
    "checkpoint_kind",
    "checkpoint_value",
    "execution_tier",
    "complete",
    "ic_mean",
    "training_hash",
    "prediction_hash",
]
catalog_rows = execution.catalog_rows.select(
    column for column in catalog_columns if column in execution.catalog_rows.columns
).sort("label", "checkpoint_value", "prediction_hash")
catalog_rows

In [ ]:
coverage_rows = []
for run in execution.runs:
    if not run.training.complete:
        raise RuntimeError(f"Incomplete training result: {run.training.hash}")
    for prediction in run.predictions:
        coverage = prediction.coverage()
        if not prediction.complete or coverage is None or coverage["status"] != "complete":
            raise RuntimeError(f"Incomplete prediction result: {prediction.hash}")
        coverage_rows.append(
            {
                "label": run.training.spec()["label"],
                "training_hash": run.training.hash,
                "prediction_hash": prediction.hash,
                "coverage_status": coverage["status"],
                "expected_rows": coverage["n_expected"],
                "actual_rows": coverage["n_actual"],
                "training_artifacts": len(run.training.artifacts()),
                "prediction_artifacts": len(prediction.artifacts()),
            }
        )

coverage_table = pl.DataFrame(coverage_rows).sort("label", "prediction_hash")
if official_population is not None:
    official_population.require_complete()
coverage_table

In [ ]:
execution_diagnostics = pl.DataFrame(execution.diagnostics)
execution_diagnostics

## Freeze the compatible result sets

A canonical default run freezes one complete IPCA result set for each configured label. Each set
is also small enough for raw diagnostic comparisons. Preview and customized canonical requests
do not publish official sets.

In [ ]:
set_rows = []
is_published_population = (
    EXECUTION_TIER == "canonical"
    and selected_labels == published_labels
    and N_FACTORS == 5
    and not OVERRIDES
)
if is_published_population:
    for selected_label in selected_labels:
        label_name = selected_label.replace("_", "-")
        result_set = study.predictions.freeze(
            execution.catalog_rows.filter(pl.col("label") == selected_label),
            name=f"us-equities-{label_name}-ipca-v1",
        )
        set_rows.append(
            {
                "role": "backtest and diagnostic population",
                "set_name": result_set.name,
                "members": len(result_set.members),
            }
        )
compatible_sets = pl.DataFrame(
    set_rows,
    schema={"role": pl.String, "set_name": pl.String, "members": pl.Int64},
)
compatible_sets

`15_model_analysis.py` reopens the named label sets for descriptive analysis. `16_backtest.py`
passes every catalog row directly to the shared backtest runner. Predictive metrics do not choose
a configuration or checkpoint.

## Key takeaways and limitations

- IPCA conditions factor loadings on the declared characteristics within each training fold.
- Convergence, fitted state, and reconstructed predictions are validated before reuse.
- The configured factor count and regularization determine the characteristic structure the model
  can represent.
- Label-specific sets preserve compatible comparisons for analysis and backtesting.